# Лабораторная работа 4. Рекомендательные системы, категориальные признаки и адаптивные методы обучения

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков IPython (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете также должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам — проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.

## Рекомендательные системы и категориальные признаки

В этой части лабораторной работы мы рассмотрим и сравним несколько различных стратегий для решения задачи рекоммендаций:
- Most popular
- Memory based
- Matrix factorization
- Categorical features based approach (без задания из-за проблем с установкой библиотеками)
- Statistics features based approach

А также научимся замешивать результаты разных стратегий в ограниченный топ с использованием Mixigen алгоритма.

Вам нужно будет провести много экспериментов, поэтому приготовьтесь.

![](https://i.imgur.com/jwyaLjP.jpg)

### Про данные

В этой лабораторной работе будет рассмотрена задача предсказания оценки, которую пользователь поставит фильму. Особенность этой задачи в том, что объекты выборки описываются категориальными признаками, принимающими большое число значений (например: идентификатор пользователя, идентификатор фильма, тэги, киноперсоны).

Мы будем работать с датасетом [MovieLens + IMDb/Rotten Tomatoes](http://files.grouplens.org/datasets/hetrec2011/), файл *hetrec2011-movielens-2k-v2.zip* (18M), [описание](http://files.grouplens.org/datasets/hetrec2011/hetrec2011-movielens-readme.txt). Набор содержит данные о предпочтениях пользователей сервиса рекомендации кинофильмов [MovieLens](http://www.movielens.org/). Пользовательские оценки для фильмов принимают целые значения в интервале от 1 до 5, они записаны в файле *user_ratedmovies.dat* (а так же в *user_ratedmovies-timestamps.dat*,  где для каждой оценки дата и время записаны в формате timestamp), остальные файлы содержат дополнительную информацию о фильмах, которую можно использовать в качестве признаков.
Заметьте: кроме оценок (и тегов), которые пользователь поставил фильмам, про пользователя ничего не известно.

На основании этих данных необходимо построить модель, предсказывающую оценку пользователя фильму, который он еще не смотрел.


- Загрузите данные и создайте разреженную матрицу оценок, где на пересечении $i$-ой строки и $j$-ого столбца стоит рейтинг, который пользователь под номером $i$ поставил фильму под номером $j$, если рейтинг известен, либо ноль, если неизвестен.
- Разбейте данные на обучающую и тестовую выборки так, чтобы в обучающую выборку вошли хронологически первые 70% всех оценок пользователей, а в тестовую хронологически последние 30%.

### Про оценку качества рекомендаций
Разбиение на train и test необходимо произвести по временной отсечке: фиксируем какое-то время в прошлом: все оценки, выставленные до этого времени, попадают в train, все оценки, выставленные после этого времени, попадают в test.

Тогда все известные рейтинги пользователя $u$ можно представить как $R^u=R^u_{train}\cup R^u_{test}$. Отсутствующие оценки обозначим за $R^u_{unknown}$. 

Выберем некоторого пользователя $u$ и обозначим известные для него рейтинги за $R^u$. 
Для измерения качества рекомендаций в этой лабораторной работе используйте две метрики RMSE и MAP, описанные ниже.

#### RMSE

Метрика [RMSE](https://en.wikipedia.org/wiki/Root-mean-square_deviation) вычисляется следующим образом:
$$ RMSE = \sqrt{ \frac{1}{|users|}\sum_{u\in users}\frac{1}{\left|R^u_{test}\right|} \sum_{i \in R^u_{test}} (r_{ui} - \hat{r}_{ui})^2 },$$
где $r_{ui}$ — наблюдаемая (правильная) оценка, а $\hat{r}_{ui}$ — оценка, предсказанная моделью; $users$ - множество всех пользователей.

Метрика RMSE предназначена для оценки точности предсказания, её удобно оптимизировать напрямую. Данная метрика - хороший выбор для задачи предсказания оценок пользователей в её непосредственной формулировке. Однако следует учесть практический аспект задачи: основная цель рекомендательной системы - рекомендовать фильмы пользователям на основе их оценок, поэтому сами значения оценок на самом деле вторичны. 

#### MAP

Для оценки качества рекомендаций можно использовать метрики качества ранжирования. В этом случае для каждого пользователя $u$ предскажем оценку для всех фильмов из $R^u_{test}$ и $R^u_{unknown}$ и отсортируем эти фильмы по убыванию предсказанного рейтинга. Ожидается, что хороший алгоритм должен выдать релевантные фильмы вверху списка. Обозначим позиции объектов в этом списке за $k^u_i$.

Назовем релевантными те фильмы, которые входят в $R^u_{test}$ и имеют оценку $\ge 3$. Обозначим их за $Rel^u$. Тогда можно вычислить следующую метрику качества рекомендаций для одного пользователя:

$$AP^u=\frac{1}{|Rel^u|} \sum_{i \in Rel^u} P@i,$$
где $$P@i = \frac{1}{i}\sum_{j=1}^i [j\in Rel^u].$$

Усреднив значение этой метрики по всем пользователями, мы получим окончательное значение метрики MAP. Пользователей без релевантных фильмов в тестовой выборке можно не учитывать.

#### Другие способы оценки качества рекомендаций

На практике, как правило, качество рекомендательных систем оценивается в онлайне с помощью [A/B-тестирования](https://en.wikipedia.org/wiki/A/B_testing).

### Most popular

**Задание 1. (1 балл).**
- Постройте рекоммендации на основе **most popular** метода, при котором пользователям рекомендуются объекты в порядке убываниях их популярности (например, среднего рейтинга).
- Оцените качество рекомендаций, вычислив RMSE и MAP.

### Memory based

Теперь рассмотрим [memory-based](https://en.wikipedia.org/wiki/Collaborative_filtering#Memory-based) методы рекоммендаций.
Подход, лежащий в их основе, использует данные о рейтингах для вычисления сходства между пользователями (user-based) или объектами (item-based), на основе этих данных делаются предсказания рейтингов и, в дальнейшем, строятся рекомендации. Эти методы просты в реализации и эффективны на ранних стадиях разработки рекомендательных систем.

Для дальнейшей работы нам понадобится библиотека [Surprise](http://surpriselib.com/). Эта библиотека заточена под оценивание рейтингов (explicit feedback).

**Задание 2. (1 балл).**
- Постройте рекомендации на основе [item-based](https://surprise.readthedocs.io/en/stable/knn_inspired.html) подхода, реализованном в библиотеке Surprise (обратите внимание на параметр *user_based* в словаре *sim_options*).
- Оцените качество рекомендаций в зависимости от выбранной функции похожести *msd/cosine/pearson* по каждой из метрик: RMSE, MAP.

### Matrix factorization

**Задание 3. (1.5 балла).**
- Разложите матрицу рейтингов с помощью [разреженного SVD](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) и, восстановив ее, получите предсказания рейтингов для всех пар пользователь-объект. В данном случае неизвестные рейтинги заполняются нулями, а затем восстанавливаются с помощью SVD (этот метод называется PureSVD).
- Рассмотрите, как минимум, 5 различных значений ранга разложения. Оцените качество рекомендаций, используя описанные выше метрики: RMSE, MAP. Для обеих метрик постройте графики зависимости качества алгоритма от выбранного ранга разложения.

- На основании полученных данных ответьте на вопросы: как значение ранга влияет на метрики и почему именно так?

**Задание 4. (2 балла).** В библиотеке Surprise также есть  алгоритмы рекомендации, основанный на матричном разложении:  [SVD, SVDpp, NMF](https://surprise.readthedocs.io/en/stable/matrix_factorization.html). Проведите эксперименты из предыдущего задания для алгоритма SVD, рассмотрев, как влияют различные параметры на результат. Сравните результаты экспериментов с полученными ранее результатами.

Мы рекомендуем при проведении экспериментов обратить внимание на следующие параметры:
 - n_factors;
 - biased;
 - набор параметров learning rate (всегда важно для алгоритма градиентого спуска);
 - набор параметров регуляризации.


#### SLIM

Пока отложем все библиотеки в сторону, пора достать с дальней полки GPU (если еще не), стряхнуть с него пыль и воспользоваться им при решении этого задания.


Более подробно алгоритм SLIM описан в [оригинальной статье](https://www.researchgate.net/publication/220765374_SLIM_Sparse_Linear_Methods_for_Top-N_Recommender_Systems). Оригинально, этот алгоритм используется именно для implicit feedback-а из бинаризации strong сигнала.

Возьмите код по SLIM с семинара и воспользуйтесь им для решения задачи кандидатогенерации/top-n-recommendation (MAP). Учтите что в базовой постановке мы не используем рейтинги, а бинаризуем их, поэтому оценивать RMSE не получится.

Бинаризируем датасет

**Задание 5. (3 балла).**

Чтобы SLIM можно было по человечески пользоваться, нашу реализацию нужно ускорить. Поищите узкие места и ускорьте код.

Несколько инсайдов по тому, где искать:
- перед оптимизацией весов для каждого айтема посчитать нормы один раз и передавать в функцию coordinate-descent
- не пересчитывать матрицу скоров для каждого j-го айтема (`R_csr @ w`), а посчитать ее один раз и инкрементально изменять
- попробуйте разные методы остановки: когда количество ненулевых компонент перестало изменяться, максимальный вес перестал изменяться и т.д.
- SLIM with Feature Selection
- и т.д.


Провалидируйте, что результат ускоренной реализации не сильно отличается от результата наивной реализации (выученные веса для ряда айтемов). Какого ускорения удалось добиться при расчете на всех айтемах?

Чтобы посчитать ускорение на всех айтемах не нужно инферинсить ни медленную ни быструю модель на всех данных, оцените их скорость приблезительно.

После ускорения вам нужно замерить качество работы вашего алгоритма. Для того чтобы это сделать, вам нужно научиться по пользователю рекомендовать товары. Воспользуйтесь для этого способом описанным в `Efficient Top-N Recommendation from SLIM` главе статьи.


**Задание 6. (2 балла).** Хоть у нас таргеты и бинаризированы, но мы можем учитывать рейтинги в развесовке семплов/айтемов. Давайте это поддержим. Есть несколько способов это учесть.

Давайте модифицируем алгоритм и посмотрим на качество рекомендаций:
- В нашей модели мы положили $r_{ui} = R_{ui} \in \{0, 1\}$, тем самым потеряв некотый сигнал о силе рейтинга уверенность в рейтинге. Положите информацию о рейтнге в матрицу, вместо бинарного значения: $r_{ui} = R_{ui} = 1 + \text{rating}_{ui} * c$, где $c=10$
- В нашей модели мы равнозрачно используем рейтинги фильмов, но это может быть плохо из-за того, что ошибки при приближении большого рейтинга более важны чем ошибки при приближении более маленького. Из-за этого предлагается ввести развесовку $c_{ui} = 1 + r_{ui} * 10$.

**Задание 7. (2 бонусных балла)** [EASE](https://arxiv.org/abs/1905.03375)

В качестве бонусного задания вам предлагаеся обучить EASE модель для кандидатогенерации. Ее можно считать расширением SLIM модели, с одним важным отличием - мы теперь работаем над dense данными, а не на sparse.

Но при этом мы материализуем не user item матрицу, которая даже для учебных датасетов может не вмещаться в память, а матрицу item-item, из-за чего получаем возможность работать с dense форматом.



### Сategorical features based approach

В этой части задания мы рассмотрим подход к рекомендациям на основе категориальных разреженных признаков. В данном случае это id-пользователя и id-фильма, а также вам будет необходимо добавить один или несколько признаков из имеющихся данных, например: жанр фильма, киноперсоны из фильма, последний оцененный пользователем фильм, средняя оценка пользователя, ...

#### Разреженные признаки

В данной части работы вам необходимо создать разреженную матрицу данных, закодировав каждый из категориальных признаков вектором чисел. Примером может служить следующая иллюстрация добавления различных признаков:

![](http://i.imgur.com/7nUMFx5.png)

Здесь, для наглядности, вся матрица объект-признак разбита на части, каждая из которых соответствует одному или группе категориальных признаков. Например, часть *User* соответствует закодированному признаку *user_id*, часть *Movie* — признаку *movie_id*, *Other movies rated* содержит в себе оценки пользователя другим фильмам, а *Last movie rated* соответствует признаку "последний оцененный пользователем фильм".

Одно из самых популярных библиотек для работы с факторизационными машинами будет [LightFM](https://github.com/lyst/lightfm), которая дает хорошие бейзлайны. Так как есть определеные сложности с установкой всех библиотек для факторизационных машин, то в этом блоке мы вас попросим только разобраться с тем, какой лосс они оптимизируют.

#### LightFM

Библиотека [LightFM](https://github.com/lyst/lightfm) реализует общий подход к задачам с категориальными признаками, в основе которого лежат факторизационные машины.

Для оптимизации она использует ранжирующий WARP лосс. В чем его идея? Разберитесь самостоятельно, это понадобится в дальнейшем.

### Statistics features based approach

В качестве иллюстрации, что такое признаки-статистики или признаки-счётчики, рассмотрим категориальный признак "жанр фильма". Для каждого жанра мы можем посчитать некоторое числовое значение, например, среднее значение оценок фильмов этого жанра. Затем, если в матрице данных мы заменим значения жанров этого категориального признака на соответствующие значения средних рейтингов данных жанров, то получим новый числовой признак-счётчик.

Здесь стоит обратить внимание на следующее:

1. При создании таких счётчиков категориальных признаков по целевой переменной, важно не использовать оценки из настоящего и будущего. То есть, в нашем примере, при расчете среднего для конкретного объекта нельзя использовать как оценку текущего объекта, так и оценки, которые были поставлены позже. Иначе возникнет переобучение.
2. В качестве счётчиков можно рассматривать и другие статистики: число встречаемости данного значения, медиану целевой переменной по объектам с тем же значением данного категориального признака, и т. п.
3. Подобные признаки-счётчики можно считать не только по одному категориальному признаку, как в примере с жанром фильма, но и и по набору из нескольких категориальных признаков, например, по паре (жанр, киноперсоны).
4. Счётчики можно считать не только по целевой переменной, но и относительно других признаков.

**Задание 8. (2.5 балла).**
- Используя исходные данные, создайте выборку с набором признаков-счётчиков.
- На полученной выборке с счетчиками постройте предсказания оценок, используя [xgboost](https://xgboost.readthedocs.io/en/stable/), [catboost](https://catboost.ai/en/docs/) или [lightGBM](https://lightgbm.readthedocs.io/en/latest/) (на ваш выбор). Постарайтесь добиться качества, сравнимого с качеством моделей из предыдущего пункта.

- Какие признаки-счётчики оказались наиболее удачными? Почему? 

**Задание 9. (0.5 балла).** Приведите сравнение качества всех моделей, использованных в работе, руководствуясь значениями описанных метрик. Какие из моделей оказались лучше других по каждой из метрик? Чем это можно объяснить?

### Mixigen

На семинаре рассматривался алгоритм Mixigen-а для замешивания различных результатов кандидатогенераторов в единый список.

Напомним, что mixigen - это алгоритм который максимизируем сумарную вероятность попадания в выдачу айтемов из кандидатогенераторов, при этом формируя список на порядки меньшего размера, чем размер списка из уникальных айтемов со всех кандидатогенераторов. Подробнее про него можно почитать [тут](https://www.linkedin.com/pulse/optimizing-candidate-generation-scale-mixigen-michael-roizner-n49se/).

Но так как у нас система упрощена, у нас есть 2 главных недостатка:
- нет информации о конверсии результатов каждого из кворумов
- формально топ, который мы берем из каждого кандидата, целиком идет в расчет метрики MAP


**Задание 10. (2 балла).** Упрощенный миксиджен

Здесь вам предлагается придумать процедуру замешивания айтемов из всех кандидатогенераторов, отталкиваясь от идеи mixigen, так чтобы при максимизировать метрику MAP при фиксированном размере топа (10/30/50).

Для этого нам нужно взять по топу из каждого кандидатогенератора так, чтобы суммарно у нас было по 10/30/50 айтемов и посчитать на них MAP метрику.

Чтобы это сделать, предлагается посчитать и воспользоваться для замешивания следующими статистиками:
- среднее позапросное количество релевантных пользователю фильмов
- долю релевантных фильмов на конкретной позиции для каждого из кворумов

Первую статистику мы можем использовать сразу для развесовки.

Вторую, с использованием [софтмакса гумбеля](https://habr.com/ru/companies/yandex/articles/834262/), для итеративного построения результирущего списка.

Воспользуйтесь этими стратегиями и посчитайте MAP@K, для K=10,30,50 для каждой из схем и сравните с аналогичными метриками но по каждому из кворумов по отдельности.

## Ранжирование

![](http://i.imgur.com/2QnD2nF.jpg)

Задачу поискового ранжирования можно описать следующим образом: имеется множество документов $d \in D$ и множество запросов $q \in Q$. Требуется оценить *степень релевантности* документа по отношению к запросу: $(q, d) \mapsto r$, относительно которой будет производиться ранжирование. Для восстановления этой зависимости используются методы машинного обучения. Обычно используется три типа:
 - признаки запроса $q$, например: мешок слов текста запроса, его длина, ...
 - документа $d$, например: значение PageRank, мешок слов, доменное имя, ...
 - пары $(q, d)$, например: число вхождений фразы из запроса $q$ в документе $d$, ...

Одна из отличительных особенностей задачи ранжирования от классических задач машинного обучения заключается в том, что качество результата зависит не от предсказанных оценок релевантности, а от порядка следования документов в рамках конкретного запроса, т.е. важно не абсолютное значение релевантности (его достаточно трудно формализовать в виде числа), а то, более или менее релевантен документ относительно других документов.


### Подходы к решению задачи ранжирования
Существуют 3 основных подхода, различие между которыми в используемой функции потерь:
  
1. **Pointwise подход**. В этом случае рассматривается *один объект* (в случае поискового ранжирования - конкретный документ) и функция потерь считается только по нему. Любой стандартный классификатор или регрессор может решать pointwise задачу ранжирования, обучившись предсказывать значение таргета. Итоговое ранжирование получается после сортировки документов к одному запросу по предсказанию такой модели.
2. **Pairwise подход**. В рамках данной модели функция потерь вычисляется по *паре объектов*. Другими словами, функция потерь штрафует модель, если отражированная этой моделью пара документов оказалась в неправильном порядке.
3. **Listwise подход**. Этот подход использует все объекты для вычисления функции потерь, стараясь явно оптимизировать правильный порядок.

### Оценка качества

Для оценивания качества ранжирования найденных документов в поиске используются асессорские оценки. Само оценивание происходит на скрытых от обучения запросах $Queries$. Для этого традиционно используется метрика *DCG* ([Discounted Cumulative Gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)) и ее нормализованный вариант — *nDCG*, всегда принимающий значения от 0 до 1.
Для одного запроса DCG считается следующим образом:
$$ DCG = \sum_{i=1}^P\frac{(2^{rel_i} - 1)}{\log_2(i+1)}, $$

где $P$ — число документов в поисковой выдаче, $rel_i$ — релевантность (асессорская оценка) документа, находящегося на i-той позиции.

*IDCG* — идеальное (наибольшее из возможных) значение *DCG*, может быть получено путем ранжирования документов по убыванию асессорских оценок.

Итоговая формула для расчета *nDCG*:

$$nDCG = \frac{DCG}{IDCG} \in [0, 1].$$

Чтобы оценить значение *nDCG* на выборке $Queries$ ($nDCG_{Queries}$) размера $N$, необходимо усреднить значение *nDCG* по всем запросам  выборки:
$$nDCG_{Queries} = \frac{1}{N}\sum_{q \in Queries}nDCG(q).$$

### Данные

Данные доступны [здесь](https://disk.yandex.ru/d/8nLlDeUFpWHOOw)


В рамках нашей задачи «документом» будет являться организация.

Разбейте обучающую выборку на обучение и контроль в соотношении 70 / 30. Обратите внимание, что разбивать необходимо множество запросов, а не строчки датасета.


Далее рассмотрим несколько подходов предсказания релевантности. Для оценивания качества моделей используйте метрику nDCG на контроле. В случае подбора гиперпараметров используйте кросс-валидацию по 5 блокам, где разбиение должно быть по запросам, а не строчкам датасета.

Пример подсчета метрики ndcg по сессиям будет представлен ниже:

In [10]:
from sklearn.metrics import ndcg_score
import numpy as np

def my_cool_ranking_algo(features):
    return np.random.random()

np.random.seed(43)

true_labels = [
    np.random.random(size=session_length)
    for session_length in np.random.randint(10, 30, size=40)
]
pred_labels = [
    np.array([my_cool_ranking_algo(...) for _ in range(session_length)])
    for session_length in map(len, true_labels)
]
np.mean([
    ndcg_score(y_true=true_label.reshape(1, -1), y_score=pred_label.reshape(1, -1))
    for true_label, pred_label in zip(true_labels, pred_labels)
])

0.8027760862768976

### Построение обучающего пула


В данных нет фичей, зато есть строковые представления запроса и объекта: отдельной задачей вам нужно представить вектор объекта по текстовому представлению запроса и названии объекта.

**Задание 11. (1 балл)**

В данном пункте вам следует закодировать все объекты набором признаков не подсматривая в тест выборку.


### Ранжируем с LinearRegression

**Задание 12. (1 балла)**

Давайте в качестве бейзлайна воспользуемся линейной регрессией, для обучения в pointwise режиме на MSE от разметки. Затем посмотрим, какой NDCG он нам дает.

### Ранжируем с торчем

Бейзлайн у нас есть, давайте теперь улучшим наши предсказания с помощью pairwise режима обучения.

**Задание 13. (2 балла)**

Реализуйте Rank-net с батчевым обновлением на обучающей выборке. В качестве модели можете взять как MLP, так и более сложную модельку - хоть предобученные енкодеры с MLP головой. Ваша задача побить по NDCG бейзлайн из линейной регрессии.



**Задание 14. (1 балл)**

Теперь реализуйте Lambda-rank с использованием NDCG для развесовки. Ваша задача получить качество не хуже, чем у алгоритма из задания 16.

**Задание 15. (1 балл)**

А теперь вспомним о предыдущей части лабы, а именно про LightFM и WARP лосс. Добавьте идею с семплированием негатива в ваш алгоритм и оцените получаемое качество алгоритма.

###  Ранжируем с CatBoost

CatBoost имеют несколько функций потерь/метрик для решения задачи ранжирования. Подробный их список с пометкой о возможности оптимизации указан в документации: https://catboost.ai/docs/en/concepts/loss-functions-ranking#usage-information

**Задание 16. (1.5 балла).** Попробуйте различные функции потерь (регрессионные () и ранжирующие), но не меньше 3, для модели CatBoost. Настройте основные параметры моделей (глубина, кол-во деревьев, скорость обучения, регуляризация).

Ваша задача получить сетап, который лучше отработает, чем ваши торчевые реализации 🤠

Сравните построенные модели с точки зрения метрики nDCG на контроле и проанализируйте полученные результаты:
  - какая модель работает лучше всего для данной задачи?
  - в чем достоинства/недостатки каждой?
  - сравните модели между собой:
   - получается ли сравнимое качество линейного pointwise подхода с остальными моделями?
   - заметна ли разница в качестве при использовании бустинга с разными функциями потерь?

**Задание 17. (1 балл).** Одним из основных преимуществ CatBoost'a является обработка категориальных факторов «из коробки». Добавьте в датасет различные категориальные факторы из данных и обучите заново CatBoost модели. Улучшилось ли качество?

**Задание 18. (3 бонусных балла)**

Для тех кто дожил до самого конца последней большой лабы есть бонус в виде рисеч задания. Здесь вам предстоит сделать обзор на 2 статьи из современного рексиса. В качестве основы для поиска статей и введение в курс дела предлагается прочитать статью с хабра))): https://habr.com/ru/companies/yandex/articles/857068/

Выберите 2 тренда, которые вам больше всего понравились и сделайте обзор ниже на каждую из статью внутри этого тренда. Во время обзора постарайтесь ограничиться парочкой абзацев и побольше картинок - проверяющему главное увидеть, что вы поняли главную идею статьи.